In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# CELL 0 — Shared prep config  (run once; used by ALL 3 datasets)

In [2]:
import os, re, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- shared output contract (identical for every dataset) ----
TARGET_SIZE  = 256                       # materialize letterboxed to 256²; online random-crop to 224² at train time
PAD_COLOR    = (0, 0, 0)                  # letterbox fill
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
OUT_ROOT     = Path("/kaggle/working/prepared")   # cleaned dataset built here, then published as a Kaggle Dataset
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}   # used when a source ships no split (rice)

# the manifest schema every dataset must emit, so the three merge cleanly:
MANIFEST_COLS = ["src_path", "filename", "label", "source_dataset", "split", "group_id"]

INPUT_ROOT = "/kaggle/input"
print("Mounted datasets:", os.listdir(INPUT_ROOT))
print("Output root     :", OUT_ROOT)

Mounted datasets: ['datasets']
Output root     : /kaggle/working/prepared


# CELL D1 — discover + merge the three cleaned manifests

In [3]:
import glob

manifest_paths = sorted(glob.glob("/kaggle/input/**/manifest.csv", recursive=True))
print("Found manifests:")
for m in manifest_paths: print("  ", m)

frames = []
for m in manifest_paths:
    mdir = Path(m).parent
    d = pd.read_csv(m)
    # resolve the CURRENT on-disk path: <manifest_dir>/images/<filename>
    d["path"] = d["filename"].apply(lambda fn: str(mdir / "images" / fn))
    frames.append(d)

merged = pd.concat(frames, ignore_index=True)

exists = merged["path"].apply(os.path.exists)
print("\nTotal rows        :", len(merged), "(expect 12859)")
print("Found on disk     :", int(exists.sum()), "| missing:", int((~exists).sum()))
print("Unique group_ids  :", merged["group_id"].nunique())

print("\nPer source_dataset:")
print(merged["source_dataset"].value_counts())
print("\nClasses:", merged["label"].nunique(), "(expect 20)")
print(merged["label"].value_counts().sort_index())

Found manifests:
   /kaggle/input/datasets/iitm21f1003346/rice-s1-cleaned-256/prepared/rice_s1/manifest.csv
   /kaggle/input/datasets/iitm21f1003346/rice-s2-cleaned-256/prepared/rice_s2/manifest.csv
   /kaggle/input/datasets/iitm21f1003346/wheat-cleaned-256/prepared/wheat/manifest.csv

Total rows        : 12859 (expect 12859)
Found on disk     : 12859 | missing: 0
Unique group_ids  : 11530

Per source_dataset:
source_dataset
wheat_kushagra3204         10673
rice_s1_nirmalsankalana     2066
rice_s2_vbookshelf           120
Name: count, dtype: int64

Classes: 20 (expect 20)
label
rice__bacterial_blight          554
rice__blast                     477
rice__brown_spot                646
rice__leaf_smut                  40
rice__tungro                    469
wheat__aphid                    859
wheat__black_rust               315
wheat__blast                    584
wheat__brown_rust              1191
wheat__common_root_rot          568
wheat__fusarium_head_blight     639
wheat__healthy     

# CELL D2 — group-aware, stratified 80/10/10 split with min-eval floor


In [5]:
TEST_FRAC, VAL_FRAC = 0.10, 0.10
MIN_EVAL = 8            # per-class floor for BOTH val and test (protects leaf_smut)

# ---- build group table: each group gets ONE representative label + its size ----
grp       = merged.groupby("group_id")
grp_label = grp["label"].agg(lambda s: s.value_counts().idxmax())   # mode label
grp_size  = grp.size()
gtab = pd.DataFrame({"label": grp_label, "size": grp_size}).reset_index()

# ---- allocate WHOLE groups per class, largest-first, to the split with the biggest deficit ----
assign = {}
for lab, sub in gtab.groupby("label"):
    n = int(sub["size"].sum())
    t_test  = max(MIN_EVAL, round(TEST_FRAC * n))
    t_val   = max(MIN_EVAL, round(VAL_FRAC * n))
    targets = {"train": n - t_test - t_val, "val": t_val, "test": t_test}
    cur     = {"train": 0, "val": 0, "test": 0}
    order = sub.sample(frac=1.0, random_state=SEED).sort_values("size", ascending=False)
    for gid, sz in zip(order["group_id"], order["size"]):
        split = max(("train", "val", "test"), key=lambda s: targets[s] - cur[s])
        assign[gid] = split
        cur[split] += int(sz)

merged["split"] = merged["group_id"].map(assign)

# ---- report ----
ct = pd.crosstab(merged["label"], merged["split"])[["train", "val", "test"]]
ct["total"] = ct.sum(axis=1)
ct["val%"]  = (ct["val"]  / ct["total"] * 100).round(1)
ct["test%"] = (ct["test"] / ct["total"] * 100).round(1)
print(ct.to_string())

print("\nOverall split sizes:", merged["split"].value_counts().to_dict())
print("Overall ratios     :", (merged["split"].value_counts(normalize=True) * 100).round(1).to_dict())

# ---- two safety checks ----
leak = merged.groupby("group_id")["split"].nunique()
print("\nLEAKAGE — groups spanning >1 split (must be 0):", int((leak > 1).sum()))
below = ct.index[(ct["val"] < MIN_EVAL) | (ct["test"] < MIN_EVAL)].tolist()
print("Classes below floor of", MIN_EVAL, "in val/test:", below or "none ✅")

split                        train  val  test  total  val%  test%
label                                                            
rice__bacterial_blight         444   55    55    554   9.9    9.9
rice__blast                    381   48    48    477  10.1   10.1
rice__brown_spot               516   65    65    646  10.1   10.1
rice__leaf_smut                 24    8     8     40  20.0   20.0
rice__tungro                   375   47    47    469  10.0   10.0
wheat__aphid                   687   86    86    859  10.0   10.0
wheat__black_rust              251   32    32    315  10.2   10.2
wheat__blast                   466   59    59    584  10.1   10.1
wheat__brown_rust              953  119   119   1191  10.0   10.0
wheat__common_root_rot         454   57    57    568  10.0   10.0
wheat__fusarium_head_blight    511   64    64    639  10.0   10.0
wheat__healthy                 828  104   104   1036  10.0   10.0
wheat__leaf_blight             455   56    56    567   9.9    9.9
wheat__mil

# CELL D3 — write final split into ImageFolder layout + master manifest

In [6]:
import shutil

FINAL_ROOT = Path("/kaggle/working/final")
if FINAL_ROOT.exists(): shutil.rmtree(FINAL_ROOT)

written, n_fail = 0, 0
for r in merged.itertuples(index=False):
    dst_dir = FINAL_ROOT / r.split / r.label
    dst_dir.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(r.path, dst_dir / r.filename)
        written += 1
    except Exception:
        n_fail += 1

# master manifest (relative paths, ready for a custom Dataset if you skip ImageFolder)
master = merged[["filename", "label", "source_dataset", "split", "group_id"]].copy()
master["rel_path"] = merged["split"] + "/" + merged["label"] + "/" + merged["filename"]
master.to_csv(FINAL_ROOT / "master_manifest.csv", index=False)

print("Images written:", written, "| failed:", n_fail)
print("Master manifest rows:", len(master))

# verify the on-disk tree matches the plan
import collections
counts = collections.Counter()
for split in ["train", "val", "test"]:
    for lab_dir in (FINAL_ROOT / split).iterdir():
        if lab_dir.is_dir():
            counts[split] += len(list(lab_dir.glob("*.jpg")))
print("On-disk per split:", dict(counts))
print("Classes in train :", len(list((FINAL_ROOT / 'train').iterdir())), "(expect 20)")

# label ↔ index map, frozen now so train/val/test share identical indices
labels_sorted = sorted(merged["label"].unique())
label_to_idx = {l: i for i, l in enumerate(labels_sorted)}
json.dump(label_to_idx, open(FINAL_ROOT / "label_to_idx.json", "w"), indent=2)
print("Wrote label_to_idx.json with", len(label_to_idx), "classes")

Images written: 12859 | failed: 0
Master manifest rows: 12859
On-disk per split: {'train': 10275, 'val': 1292, 'test': 1292}
Classes in train : 20 (expect 20)
Wrote label_to_idx.json with 20 classes
